In [ ]:
import geokit as gk
from os.path import join
import os
import numpy as np

In [ ]:
# This example showcases the capabilities of geokit to work with both raster and vector data.
# Therefore multiple publically available datasets are combined to determine the depth of offshore wind points in the North Sea.

# The three datasources that are combined for this analyses are:
# THE IHO Sea Areas, a shapefile with the sea basins of the world
# https://doi.org/10.14284/323

# The second dataset is a vector dataset with plausible wind offshore locations in the north sea by 2050
# https://zenodo.org/records/10259046

# The third dataset is the GEBCO bathymetry dataset, which is available as 15 arcsecond global raster tiles
# Because the original data is quite large we have preprocessed the to a subset of xxxx which is provided under the following link
# https://doi.org/10.5281/zenodo.17047388
# The complete original dataset can be found here:
# doi:10.5285/a29c5465-b138-234d-e053-6c86abc040b9

# 1. Download the data


# 1. Shapefile with sea basins -> filter which basin is in the North Sea; showcase the 'where' argument
# 2. Filter the GEBCO tiles that intersect with the North Sea basin -> gk.Extent.fromGeom(basins.geom).filterSources(path to the tiles)    
# 3. Combine the GEBCO tiles to a single tile -> gk.algorithms.combineSimilarRasters
# 4. Warp northsea basin onto the raster to extract only the relevant part -> gk.RegionMask.warp
# 5. Shapefile with points of offshore wind points somewhere in the North Sea -> gk.vector.extractFeatures
# 6. Interpolate the depth of the points using the GEBCO tiles -> gk.raster.interpolateValues

In [ ]:
# 1 Download the data
## Geokit provides functionality to automatically download the test data required
## Parts of the data is quite large so it might take a while to download
data_dir = join(os.path.dirname(""), 'data')


# This is the path to the sea basin data
sea_basins_path = join(data_dir, "vector", "World_Seas_IHO_v3", "World_Seas_IHO_v3.shp")

# This is the path to the vector data with offshore wind points
offshore_wind_points_path = join(data_dir, "vector", "turbine_locations shapefile", "turbine_locations.shp")


# md5:7bfb4b091c4aa457b66d75fbca361b67
# THis is 
bathymetry_rasters_dir = join(data_dir, "raster", "GEBCO_bathymetry_tiles")

import os
downloader = DOIDownloader()
https://doi.org/10.5281/zenodo.17047388
url = "doi:10.6084/m9.figshare.14763051.v1/tiny-data.txt"
# Not using with Pooch.fetch so no need to pass an instance of Pooch
downloader(url=url, output_file="tiny-data.txt", pooch=None)
os.path.exists("tiny-data.txt")

In [ ]:
# load sea basins and filter for the North Sea

sea_basins = gk.vector.extractFeatures(
    source=sea_basins_path,
    where="NAME='North Sea'"
    )

In [ ]:
# load offshore wind points
offshore_wind_points = gk.vector.extractFeatures(
    source=offshore_wind_points_path
    )

In [ ]:
## draw the sea basins and the offshore wind points
axh1 = gk.drawGeoms(sea_basins, srs=gk.srs.EPSG4326)
axh2 = gk.drawGeoms(offshore_wind_points, srs=gk.srs.EPSG4326, ax=axh1.ax)

In [ ]:
len(offshore_wind_points)

In [ ]:
## Check GEBCO tiles that overlap with the North Sea basin
bathymetry_datasets = list(gk.Extent.fromGeom(sea_basins.geom[0]).filterSources(bathymetry_rasters_dir + "/*.tif"))

In [ ]:
bathymetry_datasets

In [ ]:
# combine the GEBCO tiles to a single raster
bathymetry_raster = gk.algorithms.combineSimilarRasters(bathymetry_datasets, output="data/raster/bathymetry_raster_NorthSea.tif")

In [ ]:
GEBCO_rasterInfo = gk.raster.rasterInfo("data/raster/bathymetry_raster_NorthSea.tif")

assert GEBCO_rasterInfo.pixelHeight == GEBCO_rasterInfo.pixelWidth, "Pixel height and width must be equal. Consider warping the raster to a square pixel size."

## create a region mask
northSeaMask = gk.RegionMask.fromGeom(sea_basins.geom[0], 
                                      srs=gk.srs.EPSG4326,
                                      pixelRes=GEBCO_rasterInfo.pixelHeight
                                      )

In [ ]:
## warp the region mask onto the GEBCO raster
northSeaRasterWarped = northSeaMask.warp(source="data/raster/bathymetry_raster_NorthSea.tif",
                                        noData=GEBCO_rasterInfo.noData,
                                        returnMatrix=False,
                                        )

In [ ]:
## draw the offshore wind points on the GEBCO raster
axh = gk.drawRaster(northSeaRasterWarped, 
              srs=gk.srs.EPSG4326,
              cmap="YlGnBu", 
              figsize=(6, 6), 
              hideAxis=True,
              cbarTitle="Water Depth (m)",
              vmin=-600, 
              vmax=200,
              )

axh2 = gk.drawGeoms(sea_basins, srs=gk.srs.EPSG4326, ax=axh.ax, fc="none")
axh2 = gk.drawGeoms(offshore_wind_points, srs=gk.srs.EPSG4326, ax=axh.ax, markersize=1)

In [ ]:
## retrieve the water depth at the offshore wind points
offshore_wind_points["depth"] = offshore_wind_points.geom.apply(lambda g: gk.raster.interpolateValues(source=northSeaRasterWarped, points=g, pointSRS=gk.srs.EPSG4326))

In [ ]:
offshore_wind_points.head()